# Inspect the Catalog

Open `data/catalog.sqlite` if it exists (your Geofabrik ingest). If not, build
the committed fixture so this notebook still runs.

The notebook lists tables and columns, loads pandas DataFrames when the file
fits in RAM, then checks counts, the length band, and Trailhead locations.

A finished Catalog should look like this:

- `IngestReport.kept` equals the number of `dayhikes` rows.
- Every DayHike has a Seed label, a Trailhead, and a length in the 1–20 km band.
- Kept + unnamed + length + network drops equals the hiking relations loaded
  at ingest (`route=hiking`). Unnamed relations are kept (`dropped_unnamed` is 0);
  `name_origin` records OSM name vs ref vs `relation:{id}`.
- Trailheads sit inside the extract (Geofabrik Europe: Azores to Georgia,
  Madeira to northern Norway).
- AmenityFacts are counts inside `AMENITY_RADIUS_M` of the Trailhead, not of
  the whole continent. Parking at 20 km is common; `ele` on OSM nodes is not.


In [ ]:
import json
import os
import sqlite3
import tempfile
from pathlib import Path

from dayhike import build_catalog
from dayhike.catalog import AMENITY_RADIUS_M, MAX_LENGTH_M, MIN_LENGTH_M, OSM_ATTRIBUTION

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    if (candidate / "tests" / "fixtures" / "catalog.osm").is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
if local.is_file() and "PYTEST_CURRENT_TEST" not in os.environ:
    path = local
else:
    path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(repo / "tests" / "fixtures" / "catalog.osm", path)

conn = sqlite3.connect(path)
meta = dict(conn.execute("SELECT key, value FROM catalog_meta"))
report = json.loads(meta["ingest_report"])
print("catalog", path)
print("attribution", meta.get("attribution"))
print("ingest_report", report)


## Tables

The Catalog is seven SQLite tables. `PRAGMA table_info` below is the live
schema; this is what the columns mean.

**`catalog_meta`** — key/value. `attribution` is the ODbL line. `ingest_report`
is kept / dropped_unnamed (always 0) / dropped_length / dropped_network plus
NameOrigin counts.

**`dayhikes`** — one DayHike per row.
- `id` — OSM `route=hiking` relation id.
- `name` — Seed label: OSM `name`, else `ref`, else `relation:{id}`.
- `name_origin` — `osm_name`, `osm_ref`, or `synthetic`.
- `length_m` — assembled LineString length, 1000–20000 inclusive.
- `difficulty` — strictest member-way `sac_scale`, or NULL if untagged.

**`trailheads`** — one Trailhead per DayHike (the start point).
- `dayhike_id` — FK to `dayhikes.id`.
- `latitude`, `longitude` — WGS84. DriveTime is measured from here at query
  time; it is not a Catalog column.

**`amenity_facts`** — AmenityFacts: four rows per DayHike (`hut`, `parking`,
`restaurant_cafe`, `drinking_water`).
- `count` — how many of that kind lie inside `AMENITY_RADIUS_M` of the Trailhead.
- `nearest_m` — metres to the closest one, or NULL when count is 0.

**`feature_vectors`** — FeatureVector used for similarity. No DriveTime.
- `length_m` — same length as `dayhikes`.
- `elevation_min_m`, `elevation_max_m`, `elevation_gain_m` — from OSM `ele`
  on geometry nodes; often NULL.
- `difficulty_ordinal` — numeric `sac_scale` (hiking=1 … difficult alpine=6),
  or NULL.

**`scaler`** — StandardScaler mean and scale per FeatureVector column, fit at
ingest so ranking can standardise a query the same way.

**`labels`** — Hiker scores. Empty until someone rates a DayHike.


In [ ]:
print(f"sqlite_mb\t{path.stat().st_size / 1e6:.1f}")
tables = [
    row[0]
    for row in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1"
    )
]
for table in tables:
    n_rows = conn.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]
    print(f"table\t{table}\t{n_rows}")
    for _cid, col, ctype, notnull, _default, pk in conn.execute(
        f"PRAGMA table_info(\"{table}\")"
    ):
        print(
            f"column\t{table}\t{col}\t{ctype or 'TEXT'}\t"
            f"pk={int(pk)}\tnotnull={int(notnull)}"
        )


## pandas

Europe's Catalog is tens of megabytes; that fits in RAM. Each table becomes a
DataFrame with the same name. `play` is a wide join (DayHike + Trailhead +
FeatureVector + pivoted AmenityFacts) for filtering and plotting. Skip the
load if the SQLite file is larger than 200 MB.


In [ ]:
import pandas as pd

SQLITE_PLAY_LIMIT_MB = 200
size_mb = path.stat().st_size / 1e6
print(f"pandas sqlite_mb\t{size_mb:.1f}")
if size_mb > SQLITE_PLAY_LIMIT_MB:
    print("pandas skipped\tSQLite file larger than 200 MB")
else:
    catalog_meta = pd.read_sql_query("SELECT * FROM catalog_meta", conn)
    dayhikes = pd.read_sql_query("SELECT * FROM dayhikes", conn)
    trailheads = pd.read_sql_query("SELECT * FROM trailheads", conn)
    amenity_facts = pd.read_sql_query("SELECT * FROM amenity_facts", conn)
    feature_vectors = pd.read_sql_query("SELECT * FROM feature_vectors", conn)
    labels = pd.read_sql_query("SELECT * FROM labels", conn)
    scaler = pd.read_sql_query("SELECT * FROM scaler ORDER BY position", conn)
    amenity_wide = amenity_facts.pivot(
        index="dayhike_id", columns="kind", values=["count", "nearest_m"]
    )
    amenity_wide.columns = [
        f"{stat}_{kind}" for stat, kind in amenity_wide.columns.to_list()
    ]
    play = (
        dayhikes.rename(columns={"id": "dayhike_id"})
        .merge(trailheads, on="dayhike_id")
        .merge(feature_vectors.drop(columns=["length_m"]), on="dayhike_id")
        .merge(amenity_wide, on="dayhike_id")
    )
    for name, frame in (
        ("catalog_meta", catalog_meta),
        ("dayhikes", dayhikes),
        ("trailheads", trailheads),
        ("amenity_facts", amenity_facts),
        ("feature_vectors", feature_vectors),
        ("labels", labels),
        ("scaler", scaler),
        ("play", play),
    ):
        print(f"pandas {name}\t{frame.shape}")
    print("play dtypes")
    print(play.dtypes.to_string())
    print("play head")
    print(play.head(3).to_string())


## Counts

One Trailhead and one FeatureVector per DayHike. Labels stay empty until a
Hiker scores a hike. AmenityFacts store four kinds per DayHike (zero is valid).


In [ ]:
n = conn.execute("SELECT COUNT(*) FROM dayhikes").fetchone()[0]
n_th = conn.execute("SELECT COUNT(*) FROM trailheads").fetchone()[0]
n_fv = conn.execute("SELECT COUNT(*) FROM feature_vectors").fetchone()[0]
n_af = conn.execute("SELECT COUNT(*) FROM amenity_facts").fetchone()[0]
n_lab = conn.execute("SELECT COUNT(*) FROM labels").fetchone()[0]
empty_name = conn.execute("SELECT COUNT(*) FROM dayhikes WHERE trim(name) = ''").fetchone()[0]
dropped = (
    report["dropped_unnamed"]
    + report["dropped_length"]
    + report["dropped_network"]
)
print("dayhikes", n)
print("trailheads", n_th)
print("feature_vectors", n_fv)
print("amenity_fact_rows", n_af)
print("labels", n_lab)
print("kept_plus_dropped", report["kept"] + dropped)
print(f"sanity kept_matches_rows\t{'ok' if report['kept'] == n else 'FAIL'}")
print(f"sanity trailhead_per_dayhike\t{'ok' if n_th == n == n_fv else 'FAIL'}")
print(f"sanity attribution\t{'ok' if meta.get('attribution') == OSM_ATTRIBUTION else 'FAIL'}")
print(f"sanity named\t{'ok' if empty_name == 0 else 'FAIL'}")
print(f"sanity amenity_kinds\t{'ok' if n_af == 4 * n else 'FAIL'}")


## Length band

DayHikes are 1–20 km inclusive (`MIN_LENGTH_M` / `MAX_LENGTH_M`). Shorter and
longer hiking relations are `dropped_length`. Expect more short walks than
long ones.


In [ ]:
lo, hi, avg = conn.execute(
    "SELECT MIN(length_m), MAX(length_m), AVG(length_m) FROM dayhikes"
).fetchone()
outside = conn.execute(
    "SELECT COUNT(*) FROM dayhikes WHERE length_m < ? OR length_m > ?",
    (MIN_LENGTH_M, MAX_LENGTH_M),
).fetchone()[0]
print(f"length_m min={lo:.1f} max={hi:.1f} mean={avg:.1f}")
print(
    f"sanity length_band\t{'ok' if outside == 0 and lo >= MIN_LENGTH_M and hi <= MAX_LENGTH_M else 'FAIL'}"
)
print("length_km_histogram")
for km, count in conn.execute(
    "SELECT CAST(length_m / 1000 AS INT), COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 1"
):
    bar = "#" * max(1, round(40 * count / n)) if n else ""
    print(f"  {km:2d} km  {count:6d}  {bar}")


## Where the Trailheads are

Boxes overlap on purpose (Alps sit in France and Italy). They are a sketch,
not country membership. Extremes should be real places in the extract, not
null island.


In [ ]:
bbox = conn.execute(
    "SELECT MIN(latitude), MAX(latitude), MIN(longitude), MAX(longitude) FROM trailheads"
).fetchone()
print("trailhead_bbox lat", bbox[0], bbox[1], "lon", bbox[2], bbox[3])
print("sanity not_null_island", "ok" if bbox[0] != 0 or bbox[2] != 0 else "FAIL")

print("extremes")
for label, order in (
    ("west", "t.longitude ASC"),
    ("east", "t.longitude DESC"),
    ("south", "t.latitude ASC"),
    ("north", "t.latitude DESC"),
):
    row = conn.execute(
        "SELECT d.name, round(t.latitude, 4), round(t.longitude, 4) "
        "FROM trailheads t JOIN dayhikes d ON d.id = t.dayhike_id "
        f"ORDER BY {order} LIMIT 1"
    ).fetchone()
    print(f"  {label}\t{row[0]}\t{row[1]}\t{row[2]}")

regions = [
    ("Iberia", 36, 44, -10, 4),
    ("France", 42, 51.5, -5, 8),
    ("Alps", 44, 48.5, 5, 16),
    ("Italy", 36, 47, 6, 19),
    ("Dinaric/Balkans", 40, 47, 13, 24),
    ("Germany/Benelux", 47, 55, 2, 15),
    ("UK/IE", 49, 61, -11, 2),
    ("Scandinavia", 55, 72, 4, 32),
    ("Poland/Baltics", 49, 60, 14, 28),
    ("Romania/Carpathians", 43, 49, 20, 30),
    ("Greece", 34, 42, 19, 29),
    ("Azores/Madeira", 32, 40, -32, -13),
]
print("trailheads_in_overlapping_boxes")
for name, minlat, maxlat, minlon, maxlon in regions:
    count = conn.execute(
        "SELECT COUNT(*) FROM trailheads "
        "WHERE latitude BETWEEN ? AND ? AND longitude BETWEEN ? AND ?",
        (minlat, maxlat, minlon, maxlon),
    ).fetchone()[0]
    print(f"  {name}\t{count}")


## Amenities and elevation

Counts are inside `AMENITY_RADIUS_M` of the Trailhead (20 km). A 20 km
parking search lights up almost every European walk. Huts are sparser.
Elevation comes from `ele` on OSM nodes; most hiking geometries do not have
it, so empty elevation on the FeatureVector is expected, not a failed ingest.


In [ ]:
print("amenity_radius_m", AMENITY_RADIUS_M)
print("difficulty")
for label, count in conn.execute(
    "SELECT COALESCE(difficulty, 'untagged'), COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 2 DESC"
):
    print(f"  {label}\t{count}")
print("amenities_with_count_gt_0")
for kind, positive, avg_n, avg_near in conn.execute(
    "SELECT kind, SUM(count > 0), round(AVG(count), 1), round(AVG(nearest_m), 0) "
    "FROM amenity_facts GROUP BY kind ORDER BY kind"
):
    print(f"  {kind}\t{positive}\tmean_count={avg_n}\tmean_nearest_m={avg_near}")
elev_ok, elev_missing = conn.execute(
    "SELECT SUM(elevation_min_m IS NOT NULL), SUM(elevation_min_m IS NULL) FROM feature_vectors"
).fetchone()
print("elevation_present", elev_ok, "missing", elev_missing)


## Map

Trailheads on an Esri street map of Europe. GeoPandas draws the points;
contextily fetches tiles (needs a network). `play` is plotted when that frame
is non-empty; otherwise every Trailhead.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt

if "play" in dir() and len(play):
    plot_df = play
else:
    plot_df = trailheads.merge(
        dayhikes.rename(columns={"id": "dayhike_id"}), on="dayhike_id"
    )

gdf = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(plot_df.longitude, plot_df.latitude),
    crs="EPSG:4326",
).to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(11, 9))
gdf.plot(ax=ax, markersize=2, alpha=0.35, color="#c2410c", edgecolors="none")
if "PYTEST_CURRENT_TEST" not in os.environ:
    cx.add_basemap(ax, source=cx.providers.Esri.WorldStreetMap)
ax.set_title(f"Trailheads ({len(gdf)} of {n})")
ax.set_axis_off()
fig.tight_layout()
try:
    from IPython.display import display

    display(fig)
except ImportError:
    pass
plt.close(fig)
print("map trailheads_plotted", len(gdf), "of", n)


## Names at the length edges

Shortest and longest kept DayHikes should still read as walks, not as country
crossings. Synthetic labels look like `relation:{id}`; filter them with
`name_origin = 'synthetic'` if you want OSM names only.


In [ ]:
print("shortest")
for osm_id, name, length_m in conn.execute(
    "SELECT id, name, round(length_m) FROM dayhikes ORDER BY length_m ASC LIMIT 8"
):
    print(f"  {osm_id}\t{length_m}\t{name}")
print("longest")
for osm_id, name, length_m in conn.execute(
    "SELECT id, name, round(length_m) FROM dayhikes ORDER BY length_m DESC LIMIT 8"
):
    print(f"  {osm_id}\t{length_m}\t{name}")
print("sample")
for name in conn.execute("SELECT name FROM dayhikes ORDER BY random() LIMIT 12"):
    print(f"  {name[0]}")
conn.close()
